## Support Vector Machine (SVM)

O [SVM](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) é um modelo que busca encontrar o melhor hiperplano que separa as classes no espaço de características.

Ele pode utilizar diferentes funções de kernel para lidar com relações lineares e não lineares.

### Hiperparâmetros utilizados

- **C**: controla a margem de separação
  - Valores altos → menor margem (mais complexo)
  - Valores baixos → maior margem (mais simples)

- **kernel**: define o tipo de separação
  - `linear`: separação linear
  - `rbf`: separação não linear (default)

### Estratégia

Foi utilizado GridSearchCV com validação cruzada para identificar a melhor combinação de parâmetros, utilizando a métrica ROC AUC.

In [1]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    import pandas as pd
    import sys
    import os

    # add raiz do projeto
    sys.path.append(os.path.abspath(".."))

    from sklearn.model_selection import GridSearchCV
    from sklearn.preprocessing import StandardScaler
    from imblearn.pipeline import Pipeline
    from imblearn.over_sampling import SMOTE
    from sklearn.svm import SVC, LinearSVC
    from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

    from preprocessing.main_preprocessing import load_and_preprocess
    from utils.experiment_logger import save_results


save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade

🔎 Scenario: submodalidade_agrupada

🔎 Scenario: submodalidade_engineered


""


## BASELINE

In [2]:
# scenarios = [
#     "sem_submodalidade",
#     "submodalidade_agrupada",
#     "submodalidade_engineered"
# ]

# smote_options = [False, True]

# results = []

# for scenario in scenarios:
#     for use_smote in smote_options:

#         print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

#         X_train, X_test, y_train, y_test = load_and_preprocess(
#             "../predictive_models/scrdata_202505.csv",
#             scenario=scenario,
#             use_smote=use_smote
#         )

#         steps = []
#         steps.append(('scaler', StandardScaler()))
#         if use_smote:
#             steps.append(('smote', SMOTE(random_state=42)))
#         steps.append(('svc', LinearSVC(class_weight="balanced", random_state=42, dual=False)))

#         model = Pipeline(steps)

#         model.fit(X_train, y_train)

#         y_pred = model.predict(X_test)
#         y_proba = model.decision_function(X_test)

#         results.append({
#             "model": "SVM",
#             "scenario": scenario,
#             "smote": use_smote,
#             "roc_auc": roc_auc_score(y_test, y_proba),
#             "f1": f1_score(y_test, y_pred),
#             "accuracy": accuracy_score(y_test, y_pred),
#             "n_features": X_train.shape[1],
#             "phase": "baseline",
#             "timestamp": pd.Timestamp.now()
#         })


# save_results(results, file_path="../results/experiments.csv")

# display(pd.DataFrame(results))


## GRIDSEARCH

In [3]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    X_train, X_test, y_train, y_test = load_and_preprocess(
        "../predictive_models/scrdata_202505.csv",
        scenario=scenario,
        use_smote=False
    )


    param_grid_svm = {
        "smote": [SMOTE(random_state=42), "passthrough"],
        "svc__C": [0.1, 1, 10]
    }


    grid_svm = GridSearchCV(
        estimator=Pipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=42)),
            ('svc', LinearSVC(class_weight="balanced", random_state=42, dual=False))
        ]),
        param_grid=param_grid_svm,
        cv=5,                
        scoring="roc_auc",
        n_jobs=2
    )

    grid_svm.fit(X_train, y_train)

    print("Best parameters:", grid_svm.best_params_)
    print("Best ROC AUC (CV):", grid_svm.best_score_)


    #? BEST MODEL TEST EVALUATION

    best_svm = grid_svm.best_estimator_

    y_pred = best_svm.predict(X_test)
    y_proba = best_svm.decision_function(X_test)


    #? TUNING (CV)



    cv_results = pd.DataFrame(grid_svm.cv_results_)
    cv_results['smote_used'] = cv_results['param_smote'].apply(lambda x: x != 'passthrough')

    for smote_val in [True, False]:
        sub_results = cv_results[cv_results['smote_used'] == smote_val]
        if not sub_results.empty:
            best_row = sub_results.sort_values('mean_test_score', ascending=False).iloc[0]
            params = best_row['params']

            tuning_results.append({
                "model": "SVM",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "tuning_cv",
                "roc_auc": best_row['mean_test_score'],
                "f1": None,
                "accuracy": None,
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

            # Re-fit the best model for this group to get test metrics
            best_model_group = grid_svm.estimator.set_params(**params)
            best_model_group.fit(X_train, y_train)

            y_pred = best_model_group.predict(X_test)
            y_proba = best_model_group.decision_function(X_test)

            tuning_results.append({
                "model": "SVM",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "test",
                "roc_auc": roc_auc_score(y_test, y_proba),
                "f1": f1_score(y_test, y_pred),
                "accuracy": accuracy_score(y_test, y_pred),
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

    save_results(tuning_results, file_path="../results/model_results.csv")

    display(pd.DataFrame(tuning_results))

save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade
Best parameters: {'smote': SMOTE(random_state=42), 'svc__C': 10}
Best ROC AUC (CV): 0.8640817237777805


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,SVM,sem_submodalidade,True,tuning_cv,0.864082,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:12:00.945022
1,SVM,sem_submodalidade,True,test,0.864030,0.804102,0.785910,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:12:12.469865
2,SVM,sem_submodalidade,False,tuning_cv,0.864007,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:12:12.470633
3,SVM,sem_submodalidade,False,test,0.863847,0.804301,0.786074,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:12:21.728245



🔎 Scenario: submodalidade_agrupada
Best parameters: {'smote': 'passthrough', 'svc__C': 10}
Best ROC AUC (CV): 0.9103768068981323


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,SVM,sem_submodalidade,True,tuning_cv,0.864082,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:12:00.945022
1,SVM,sem_submodalidade,True,test,0.864030,0.804102,0.785910,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:12:12.469865
2,SVM,sem_submodalidade,False,tuning_cv,0.864007,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:12:12.470633
3,SVM,sem_submodalidade,False,test,0.863847,0.804301,0.786074,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:12:21.728245
4,SVM,submodalidade_agrupada,True,tuning_cv,0.910376,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:16:27.943393
5,SVM,submodalidade_agrupada,True,test,0.911029,0.845540,0.828673,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:16:48.483843
6,SVM,submodalidade_agrupada,False,tuning_cv,0.910377,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:16:48.484561
7,SVM,submodalidade_agrupada,False,test,0.910996,0.843766,0.827213,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:17:03.502844



🔎 Scenario: submodalidade_engineered
Best parameters: {'smote': SMOTE(random_state=42), 'svc__C': 10}
Best ROC AUC (CV): 0.8640817237777805


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,SVM,sem_submodalidade,True,tuning_cv,0.864082,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:12:00.945022
1,SVM,sem_submodalidade,True,test,0.864030,0.804102,0.785910,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:12:12.469865
2,SVM,sem_submodalidade,False,tuning_cv,0.864007,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:12:12.470633
3,SVM,sem_submodalidade,False,test,0.863847,0.804301,0.786074,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:12:21.728245
4,SVM,submodalidade_agrupada,True,tuning_cv,0.910376,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:16:27.943393
5,SVM,submodalidade_agrupada,True,test,0.911029,0.845540,0.828673,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:16:48.483843
6,SVM,submodalidade_agrupada,False,tuning_cv,0.910377,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:16:48.484561
7,SVM,submodalidade_agrupada,False,test,0.910996,0.843766,0.827213,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:17:03.502844
8,SVM,submodalidade_engineered,True,tuning_cv,0.864082,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:19:56.022620
9,SVM,submodalidade_engineered,True,test,0.864030,0.804102,0.785910,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:20:08.363746


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,SVM,sem_submodalidade,True,tuning_cv,0.864082,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:12:00.945022
1,SVM,sem_submodalidade,True,test,0.864030,0.804102,0.785910,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:12:12.469865
2,SVM,sem_submodalidade,False,tuning_cv,0.864007,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:12:12.470633
3,SVM,sem_submodalidade,False,test,0.863847,0.804301,0.786074,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:12:21.728245
4,SVM,submodalidade_agrupada,True,tuning_cv,0.910376,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:16:27.943393
5,SVM,submodalidade_agrupada,True,test,0.911029,0.845540,0.828673,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:16:48.483843
6,SVM,submodalidade_agrupada,False,tuning_cv,0.910377,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:16:48.484561
7,SVM,submodalidade_agrupada,False,test,0.910996,0.843766,0.827213,"{'smote': 'passthrough', 'svc__C': 10}",2026-05-20 21:17:03.502844
8,SVM,submodalidade_engineered,True,tuning_cv,0.864082,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:19:56.022620
9,SVM,submodalidade_engineered,True,test,0.864030,0.804102,0.785910,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-05-20 21:20:08.363746
